# Emotions dataset -> sentiment LTS pipeline (GitHub + Colab)

**Status:** This notebook references `src/main_cluster_sentiment.py` and `src/eval_sentiment.py`, which are **not included** in the current `emotions-rec-sentiments` tree. Training/eval cells will fail until those scripts exist.

**Use instead:** `notebooks/emotions_rec_sentiment_pipeline.ipynb` for the **binary** (e.g. love vs rest) pipeline that *does* ship here.

---

This notebook mirrors the same step-by-step style as the existing repro notebook, but was written for a sentiment-only pipeline on branch `emotions-rec-sentiments`.

Sentiment mapping:
- `0 negative` = sadness, anger, fear
- `1 neutral` = surprise
- `2 positive` = joy, love


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import sys

REPO_URL = "https://github.com/ShenyaoZhang/DS-GA-3001-Data-Engineering-Project.git"
BRANCH = "emotions-rec-sentiments"

DRIVE_ROOT = "/content/drive/MyDrive"
REPO_ROOT = os.path.join(DRIVE_ROOT, "DS-GA-3001-Data-Engineering-Project")
EXPERIMENT_ROOT = os.path.join(REPO_ROOT, "emotions_rec")

SRC_DIR = os.path.join(EXPERIMENT_ROOT, "src")
PROMPTS_DIR = os.path.join(EXPERIMENT_ROOT, "prompts")
DATA_PROCESSED_DIR = os.path.join(EXPERIMENT_ROOT, "data", "processed")

if not os.path.exists(REPO_ROOT):
    !git clone -b {BRANCH} {REPO_URL} "{REPO_ROOT}"
else:
    %cd "{REPO_ROOT}"
    !git fetch origin
    !git checkout {BRANCH}
    !git pull

%cd "{EXPERIMENT_ROOT}"
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("Experiment root:", EXPERIMENT_ROOT)


In [ ]:
from getpass import getpass
import os

token = getpass("Enter your HF token: ")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token
print("HF token set.")


In [ ]:
!pip -q install -r "{os.path.join(REPO_ROOT, 'requirements.txt')}"
!pip -q install datasets


In [ ]:
import csv
import json
import pandas as pd
from datasets import load_dataset

EMOTION_NAMES = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
EMOTION_TO_SENTIMENT = {0: 0, 1: 2, 2: 2, 3: 0, 4: 0, 5: 1}

ds = load_dataset("dair-ai/emotion")

def split_to_df(split_data, id_offset=0):
    rows = []
    for i, ex in enumerate(split_data):
        rows.append({
            "id": id_offset + i,
            "text": str(ex["text"]).strip(),
            "raw_label": int(ex["label"]),
        })
    return pd.DataFrame(rows)

train_raw = split_to_df(ds["train"], 0)
val_raw = split_to_df(ds["validation"], len(ds["train"]))
test_raw = split_to_df(ds["test"], len(ds["train"]) + len(ds["validation"]))

def to_sentiment(df):
    out = df.copy()
    out["label"] = out["raw_label"].map(EMOTION_TO_SENTIMENT)
    out["emotion"] = out["raw_label"].map(EMOTION_NAMES)
    out["title"] = out["text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    return out[["id", "title", "label", "emotion"]]

train_lts = to_sentiment(train_raw)
val_lts = to_sentiment(val_raw)
test_lts = to_sentiment(test_raw)

TARGET_SLUG = "sentiment"
train_path = os.path.join(DATA_PROCESSED_DIR, f"train_inner_emotions_{TARGET_SLUG}.csv")
val_path = os.path.join(DATA_PROCESSED_DIR, f"val_emotions_{TARGET_SLUG}.csv")
test_path = os.path.join(DATA_PROCESSED_DIR, f"test_emotions_{TARGET_SLUG}.csv")

train_lts.to_csv(train_path, index=False, quoting=csv.QUOTE_ALL)
val_lts.to_csv(val_path, index=False, quoting=csv.QUOTE_ALL)
test_lts.to_csv(test_path, index=False, quoting=csv.QUOTE_ALL)

print(train_path)
print(train_lts["label"].value_counts().sort_index())


In [ ]:
def build_few_shot(df, n_per_class=3):
    examples = []
    for cls in sorted(df["label"].unique()):
        part = df[df["label"] == cls].head(n_per_class)
        for _, row in part.iterrows():
            examples.append({
                "id": int(row["id"]),
                "text": str(row["title"])[:140],
                "label": str(int(row["label"])),
            })
    return examples

few_shot = build_few_shot(train_lts)
few_shot_path = os.path.join(PROMPTS_DIR, "few_shot_examples_sentiment.json")
with open(few_shot_path, "w", encoding="utf-8") as f:
    json.dump(few_shot, f, indent=2, ensure_ascii=False)

print("Saved:", few_shot_path)
print("Num examples:", len(few_shot))


In [ ]:
# Files that actually ship on emotions-rec-sentiments (binary pipeline)
SHIPPED = [
    "preprocessing.py",
    "LDA.py",
    "labeling.py",
    "random_sampling.py",
    "thompson_sampling.py",
    "fine_tune.py",
    "main_cluster_emotion_binary.py",
    "eval_emotion_binary.py",
]

# 3-class sentiment drivers (this branch does not include them)
SENTIMENT_OPTIONAL = ["sentiment_labels.py", "main_cluster_sentiment.py", "eval_sentiment.py"]

missing = []
for name in SHIPPED:
    path = os.path.join(SRC_DIR, name)
    ok = os.path.exists(path)
    print(f"{name}: {'FOUND' if ok else 'MISSING'}")
    if not ok:
        missing.append(name)

print("\nOptional (sentiment 3-class — usually absent on this branch):")
for name in SENTIMENT_OPTIONAL:
    path = os.path.join(SRC_DIR, name)
    print(f"  {name}: {'FOUND' if os.path.exists(path) else 'absent'}")

if missing:
    raise FileNotFoundError(f"Missing shipped files: {missing}")

EXPECTED = SHIPPED  # used by py_compile cell below


In [ ]:
import subprocess

targets = [os.path.join(SRC_DIR, f) for f in EXPECTED]
res = subprocess.run(["python", "-m", "py_compile", *targets], capture_output=True, text=True, cwd=EXPERIMENT_ROOT)
if res.returncode == 0:
    print("All files compiled successfully.")
else:
    print(res.stderr)


In [ ]:
# This branch does not include main_cluster_sentiment.py. Do not run the old command here.
print(
    "Skip sentiment training in this notebook.\n"
    "Use notebooks/emotions_rec_sentiment_pipeline.ipynb (binary love vs rest), or add\n"
    "main_cluster_sentiment.py / eval_sentiment.py to this repo and rebuild data CSVs."
)


In [ ]:
# Optional: inspect the latest training log file
import glob

logs = sorted(glob.glob("outputs/sentiment_train_*.log"))
if logs:
    print("Latest log:", logs[-1])
    !tail -n 80 "{logs[-1]}"
else:
    print("No sentiment log file found yet.")

## Evaluate the trained sentiment model

After training, run this cell with your best saved checkpoint directory from `models/`.

It prints:
- per-class precision/recall/F1
- accuracy
- macro/weighted F1
- mean prediction confidence

In [ ]:
# eval_sentiment.py is not in this branch. For binary eval use eval_emotion_binary.py (see README.md).
print(
    "Example (after binary training):\n"
    "  python src/eval_emotion_binary.py \\\n"
    "    -test_path data/processed/emotions_love_test.csv \\\n"
    "    -model_path models/<your_checkpoint_dir> \\\n"
    "    -target_emotion love"
)